# Feature Engineering & Preprocessing Pipeline (Leak-Free)

**Goal:** a reusable scikit-learn pipeline for a mixed-type tabular dataset (Credit Default Risk) with encoding, scaling, missing-value imputation, correlation analysis and feature-importance ranking.

**Leakage rule:** the train/test split happens **first**. Every imputer, scaler, encoder and statistic is fit on the training set only.

Dataset: German Credit (`credit-g`, OpenML), target = credit risk (1 = bad/default). If OpenML is unreachable, the notebook falls back to a synthetic credit-like dataset and says so in the output.

In [1]:
import warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

In [2]:
# ---- 1. Load a complex tabular dataset ----
def load_data():
    try:
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("credit-g", version=1, as_frame=True, return_X_y=True)
        X = X.copy()
        for c in X.columns:                       # category dtype -> plain object
            if str(X[c].dtype) == "category":
                X[c] = X[c].astype(object)
        y = (y.astype(str).str.lower() == "bad").astype(int)   # 1 = bad credit (default)
        return X, y, "OpenML credit-g (German Credit)"
    except Exception as e:
        print("OpenML unavailable (%s); using synthetic credit-like data" % type(e).__name__)
        n = 1000
        X = pd.DataFrame({
            "age": rng.integers(19, 75, n).astype(float),
            "duration": rng.integers(4, 72, n).astype(float),
            "credit_amount": rng.lognormal(7.8, 0.8, n).round(),
            "installment_rate": rng.integers(1, 5, n).astype(float),
            "existing_credits": rng.integers(1, 5, n).astype(float),
            "checking_status": rng.choice(["<0", "0<=X<200", ">=200", "none"], n),
            "credit_history": rng.choice(["critical", "existing paid", "delayed", "all paid", "no credits"], n),
            "purpose": rng.choice(["car", "radio/tv", "furniture", "business", "education", "repairs",
                                   "new car", "used car", "vacation", "other"], n),
            "savings_status": rng.choice(["<100", "100-500", "500-1000", ">=1000", "unknown"], n),
            "employment": rng.choice(["unemployed", "<1", "1-4", "4-7", ">=7"], n),
            "housing": rng.choice(["own", "rent", "free"], n),
            "job": rng.choice(["unskilled", "skilled", "management", "unemployed"], n)})
        z = (-1.5 + 0.03 * X.duration + 0.0002 * X.credit_amount - 0.02 * X.age
             + (X.checking_status == "<0") * 0.9 + (X.credit_history == "critical") * -0.5)
        y = pd.Series((rng.random(n) < 1 / (1 + np.exp(-z))).astype(int), name="default")
        return X, y, "SYNTHETIC credit-like data (fallback)"

X, y, SOURCE = load_data()
print("Source:", SOURCE, "| shape:", X.shape, "| default rate: %.1f%%" % (100 * y.mean()))
X.head()

Source: OpenML credit-g (German Credit) | shape: (1000, 20) | default rate: 30.0%


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes


In [3]:
# ---- 2. Optional: mask 5% of values so the imputers have real work ----
# The raw credit dataset is complete; this is a clearly labelled demonstration of the missing-value pipeline.
DEMO_MISSING, MISS_RATE = True, 0.05
if DEMO_MISSING:
    for c in X.columns:
        X.loc[rng.random(len(X)) < MISS_RATE, c] = np.nan
print("Missing values per column (top 5):\n", X.isna().sum().sort_values(ascending=False).head().to_string())

Missing values per column (top 5):
 other_parties             58
installment_commitment    55
existing_credits          55
credit_history            54
num_dependents            54


In [4]:
# ---- 3. Train/test split BEFORE any transformation ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (800, 20) Test: (200, 20)


In [5]:
# ---- 4. Reusable building blocks ----
class LogSkewed(BaseEstimator, TransformerMixin):
    """Applies log1p to non-negative, strongly skewed columns. Skew is measured on TRAIN data only."""
    def __init__(self, skew_threshold=1.0):
        self.skew_threshold = skew_threshold
    def _remember(self, X):
        n = np.asarray(X).shape[1]
        names = list(X.columns) if hasattr(X, "columns") else [f"x{i}" for i in range(n)]
        self.names_ = [str(c) for c in names]
    def fit(self, X, y=None):
        self._remember(X)
        df = pd.DataFrame(np.asarray(X, dtype=float))
        self.cols_ = [i for i in range(df.shape[1])
                      if df.iloc[:, i].min() >= 0 and abs(df.iloc[:, i].skew()) > self.skew_threshold]
        return self
    def transform(self, X):
        out = np.asarray(X, dtype=float).copy()
        for i in self.cols_:
            out[:, i] = np.log1p(np.clip(out[:, i], 0, None))
        if hasattr(X, "columns"):
            self._remember(X)
            return pd.DataFrame(out, columns=X.columns, index=X.index)
        return out
    def get_feature_names_out(self, input_features=None):
        if input_features is not None and not (np.ndim(input_features) == 0):
            return np.asarray(input_features, dtype=object)
        return np.asarray(self.names_, dtype=object)

def build_preprocessor(X_train, high_card_threshold=8):
    """Column types are inferred from the TRAIN frame; returns an unfitted ColumnTransformer."""
    num_cols = X_train.select_dtypes(include="number").columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]
    low = [c for c in cat_cols if X_train[c].nunique() <= high_card_threshold]
    high = [c for c in cat_cols if X_train[c].nunique() > high_card_threshold]
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                         ("log", LogSkewed()), ("scale", StandardScaler())])
    low_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    high_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("target", TargetEncoder(target_type="binary", random_state=SEED))])
    ct = ColumnTransformer([("num", num_pipe, num_cols), ("onehot", low_pipe, low),
                            ("target", high_pipe, high)], remainder="drop", verbose_feature_names_out=True)
    ct.set_output(transform="pandas")
    return ct, {"numeric": num_cols, "onehot": low, "target_encoded": high}

pre, groups = build_preprocessor(X_train)
for k, v in groups.items():
    print("%-15s (%d): %s" % (k, len(v), v))

numeric         (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
onehot          (12): ['checking_status', 'credit_history', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
target_encoded  (1): ['purpose']


In [6]:
# ---- 5. Correlation analysis (TRAIN only) ----
num = groups["numeric"]
corr = X_train[num].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=60, ha="right")
ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
fig.colorbar(im); ax.set_title("Numeric feature correlation (train)")
plt.tight_layout(); plt.savefig("correlation_heatmap.png", dpi=120)
pairs = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(num) for b in num[i + 1:] if abs(corr.loc[a, b]) > 0.7]
print("Highly correlated pairs (|r| > 0.7):", pairs or "none")
print("\nCorrelation with target (train):")
print(X_train[num].corrwith(y_train).sort_values(key=abs, ascending=False).round(3).to_string())

Highly correlated pairs (|r| > 0.7): none

Correlation with target (train):
duration                  0.206
credit_amount             0.143
installment_commitment    0.076
age                      -0.063
existing_credits         -0.035
residence_since          -0.011
num_dependents            0.008


In [7]:
# ---- 6. Fit on TRAIN, transform both; prove no leakage ----
Xt_train = pre.fit_transform(X_train, y_train)
Xt_test = pre.transform(X_test)
print("Transformed train:", Xt_train.shape, "| test:", Xt_test.shape)
print("Same columns in train and test:", list(Xt_train.columns) == list(Xt_test.columns))
print("Missing values after pipeline: train=%d, test=%d" % (Xt_train.isna().sum().sum(), Xt_test.isna().sum().sum()))

scaler = pre.named_transformers_["num"].named_steps["scale"]
all_mean = pd.concat([X_train, X_test])[groups["numeric"]].median().values
print("\nScaler was fit on %d train rows only (n_samples_seen_ = %d of %d total)" % (
    len(X_train), scaler.n_samples_seen_, len(X)))
assert scaler.n_samples_seen_ == len(X_train), "Scaler saw non-train rows!"
num_cols_out = [c for c in Xt_train.columns if c.startswith("num__") and "missingindicator" not in c]
print("Train numeric mean ~0: %.3f | Test numeric mean: %.3f (not exactly 0, as expected)" % (
    Xt_train[num_cols_out].mean().abs().mean(), Xt_test[num_cols_out].mean().abs().mean()))
Xt_train.head()

Transformed train: (800, 59) | test: (200, 59)
Same columns in train and test: True
Missing values after pipeline: train=0, test=0

Scaler was fit on 800 train rows only (n_samples_seen_ = 800 of 1000 total)
Train numeric mean ~0: 0.000 | Test numeric mean: 0.067 (not exactly 0, as expected)


,num__duration,num__credit_amount,num__installment_commitment,num__residence_since,num__age,num__existing_credits,num__num_dependents,num__missingindicator_duration,num__missingindicator_credit_amount,num__missingindicator_installment_commitment,...,onehot__housing_rent,onehot__job_high qualif/self emp/mgmt,onehot__job_skilled,onehot__job_unemp/unskilled non res,onehot__job_unskilled resident,onehot__own_telephone_none,onehot__own_telephone_yes,onehot__foreign_worker_no,onehot__foreign_worker_yes,target__purpose
828,1.277378,1.667869,0.042638,1.068901,1.151848,-0.716528,-0.418023,-0.207424,-0.207424,-0.258199,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.177715
997,-0.704341,-1.434459,0.964537,1.068901,0.406107,-0.716528,-0.418023,-0.207424,-0.207424,-0.258199,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.228021
148,1.277378,1.084719,0.042638,-0.790057,-0.657939,1.188057,-0.418023,-0.207424,-0.207424,-0.258199,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.354584
735,1.277378,0.690330,0.042638,-0.790057,-0.536181,-0.716528,-0.418023,-0.207424,-0.207424,-0.258199,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.410211
130,1.809582,1.691851,-1.801161,-0.790057,-1.190993,-0.716528,-0.418023,-0.207424,-0.207424,-0.258199,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.357201


In [8]:
# ---- 7. Feature importance: mutual information + random-forest ----
mi = pd.Series(mutual_info_classif(Xt_train, y_train, random_state=SEED), index=Xt_train.columns)
rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xt_train, y_train)
rf_imp = pd.Series(rf.feature_importances_, index=Xt_train.columns)
imp = pd.DataFrame({"mutual_info": mi, "rf_importance": rf_imp})
imp["rank_mi"] = imp.mutual_info.rank(ascending=False)
imp["rank_rf"] = imp.rf_importance.rank(ascending=False)
imp["avg_rank"] = imp[["rank_mi", "rank_rf"]].mean(axis=1)
imp = imp.sort_values("avg_rank")
print(imp.head(15).round(4).to_string())
top = imp.head(12)[::-1]
fig, axs = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axs[0].barh(top.index, top.mutual_info); axs[0].set_title("Mutual information")
axs[1].barh(top.index, top.rf_importance); axs[1].set_title("Random-forest importance")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=120)
imp.round(5).to_csv("feature_importance.csv")

                                                       mutual_info  rf_importance  rank_mi  rank_rf  avg_rank
num__duration                                               0.0359         0.0708      3.0      3.0       3.0
onehot__checking_status_no checking                         0.0698         0.0499      1.0      5.0       3.0
num__residence_since                                        0.0293         0.0309      5.0      8.0       6.5
num__credit_amount                                          0.0145         0.0942     16.0      1.0       8.5
onehot__employment_<1                                       0.0370         0.0167      2.0     15.0       8.5
onehot__property_magnitude_real estate                      0.0247         0.0174      6.0     13.0       9.5
onehot__credit_history_critical/other existing credit       0.0178         0.0208     10.0     10.0      10.0
onehot__other_payment_plans_none                            0.0200         0.0167      7.0     14.0      10.5
target__pu

In [9]:
# ---- 8. Sanity check: the whole pipeline inside cross-validation (no leakage) ----
full = Pipeline([("pre", build_preprocessor(X_train)[0]), ("clf", LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
scores = cross_val_score(full, X_train, y_train, cv=cv, scoring="roc_auc")
print("5-fold ROC-AUC on train (preprocessing refit inside each fold): %.3f +/- %.3f" % (scores.mean(), scores.std()))
full.fit(X_train, y_train)
from sklearn.metrics import roc_auc_score
print("Held-out test ROC-AUC: %.3f" % roc_auc_score(y_test, full.predict_proba(X_test)[:, 1]))

5-fold ROC-AUC on train (preprocessing refit inside each fold): 0.777 +/- 0.048
Held-out test ROC-AUC: 0.746


In [10]:
# ---- 9. Save the reusable pipeline ----
joblib.dump(pre, "preprocessing_pipeline.joblib")
print("Saved preprocessing_pipeline.joblib. Reuse: pre = joblib.load(...); pre.transform(new_df)")

Saved preprocessing_pipeline.joblib. Reuse: pre = joblib.load(...); pre.transform(new_df)
